# Optional: CPU and WebGPU measurements

Run this notebook when you want to compare performance or check the compiled output
against eager PyTorch. Use a fresh kernel for each device and run one notebook at a
time. The camera demos do not run these comparisons.

Change `DEVICE` below to `cpu` or `webgpu`. The first prediction includes compilation,
execution and readback; warmed prediction and output-copy timings are separate.

Saved outputs below come from a local browser run. Rerun to measure your own device.


In [1]:
import base64
import json
import time
import statistics
from pathlib import Path

import cv2
import numpy as np
import torch
from IPython.display import Markdown, display
from modnet_models.modnet import MODNet

DEVICE = 'webgpu'
SIZE = 256
torch.autograd.set_multithreading_enabled(False)
print(f'PyTorch {torch.__version__} | OpenCV {cv2.__version__} | {DEVICE}')


PyTorch 2.15.0a0 | OpenCV 5.0.0 | webgpu


## 1. Load the pretrained model

MODNet predicts a soft foreground mask around a person, including partially
transparent edges such as hair. We use 256×256 inputs for the live preview.


In [2]:
from modnet_weights import load_checkpoint

model = None
load_started = time.perf_counter()
checkpoint = await load_checkpoint()
model = MODNet(backbone_pretrained=False).eval()
model.load_state_dict({name.removeprefix('module.'): value
                       for name, value in checkpoint.items()}, strict=True)
del checkpoint
for parameter in model.parameters():
    parameter.requires_grad_(False)
model_load_seconds = time.perf_counter() - load_started
print(f'Pretrained model ready: {sum(p.numel() for p in model.parameters()):,} parameters')

def prepare_image(bgr):
    rgb = cv2.cvtColor(cv2.resize(bgr, (SIZE, SIZE), interpolation=cv2.INTER_LINEAR),
                       cv2.COLOR_BGR2RGB)
    normalized = rgb.astype(np.float32) / 127.5 - 1
    return torch.from_numpy(normalized.transpose(2, 0, 1).copy()).unsqueeze(0)

def predict_matte(image):
    return model(image, True)[2]

sample = cv2.imread('astronaut.png')
assert sample is not None
example = prepare_image(sample)
eager_started = time.perf_counter()
with torch.inference_mode():
    expected = predict_matte(example)
eager_reference_seconds = time.perf_counter() - eager_started
assert torch.isfinite(expected).all()
print('Eager reference ready:', tuple(expected.shape))


Loading bundled pretrained weights.
Pretrained model ready: 6,487,075 parameters
Eager reference ready: (1, 1, 256, 256)


## 2. Compile and check the model

Compile on the selected device with `torch.compile`, then check its prediction
against the eager CPU reference. The first call includes compilation.


In [3]:
device_setup_started = time.perf_counter()
if DEVICE == 'webgpu':
    from iree.runtime.webgpu import get_webgpu_device
    from iree.turbine.runtime.torch_device import TorchDevice
    gpu = TorchDevice(await get_webgpu_device(
        max_concurrent_operations=len(model.state_dict()) + 1), name='webgpu')
    backend = gpu.backend
else:
    gpu = None
    backend = 'turbine_cpu'

model = model.to(DEVICE)
example_device = example.to(DEVICE)
if gpu is not None:
    await gpu.synchronize()
device_setup_seconds = time.perf_counter() - device_setup_started
compiled_model = torch.compile(predict_matte, backend=backend,
                               fullgraph=True, dynamic=False)

started = time.perf_counter()
with torch.inference_mode():
    alpha = compiled_model(example_device)
actual = await gpu.to_cpu(alpha) if gpu is not None else alpha.detach()
cold_seconds = time.perf_counter() - started
torch.testing.assert_close(actual, expected, rtol=3e-3, atol=8e-4)
report = dict(passed=True, device=DEVICE, input_size=SIZE, opencv=cv2.__version__,
              cold_first_call_seconds=cold_seconds,
              max_abs_error=float((actual - expected).abs().max()))
print(json.dumps(report, indent=2))


/lib/python3.14/site-packages/torch/utils/_filelock.py:4: UserWarning: only soft file lock is available
  from filelock import FileLock as base_FileLock


{
  "passed": true,
  "device": "webgpu",
  "input_size": 256,
  "opencv": "5.0.0",
  "cold_first_call_seconds": 12.901,
  "max_abs_error": 7.149577140808105e-05
}


## 3. CPU or WebGPU timings

Run once with `DEVICE = 'cpu'`, then again with `DEVICE = 'webgpu'` in a fresh
kernel. The first prediction includes compilation, execution and readback. Warmed
predictions reuse the input on the selected device; output copying is measured
separately. These are inference timings. Each run saves its own JSON report.


In [4]:
repeats = 10
completed_seconds = []
with torch.inference_mode():
    for _ in range(repeats):
        started = time.perf_counter()
        prediction = compiled_model(example_device)
        if gpu is not None:
            await gpu.synchronize()
        completed_seconds.append(time.perf_counter() - started)

started = time.perf_counter()
checked = await gpu.to_cpu(prediction) if gpu is not None else prediction.detach().clone()
readback_seconds = time.perf_counter() - started
torch.testing.assert_close(checked, expected, rtol=3e-3, atol=8e-4)

metrics = dict(device=DEVICE, model_load_seconds=model_load_seconds,
               eager_cpu_reference_seconds=eager_reference_seconds,
               device_setup_seconds=device_setup_seconds,
               compile_and_first_prediction_seconds=cold_seconds,
               resident_inference_median_ms=statistics.median(completed_seconds) * 1000,
               output_copy_or_readback_ms=readback_seconds * 1000,
               repeats=repeats)
report.update(metrics)
display(Markdown(
    f"| {DEVICE.upper()} measurement | Time |\n| :--- | ---: |\n"
    f"| Load pretrained model | {model_load_seconds:.3f} s |\n"
    f"| Eager reference (CPU) | {eager_reference_seconds:.3f} s |\n"
    f"| Device setup and initial upload | {device_setup_seconds:.3f} s |\n"
    f"| Compile and complete the first prediction | {cold_seconds:.3f} s |\n"
    f"| Warmed prediction, resident input (median of {repeats}) | "
    f"{metrics['resident_inference_median_ms']:.2f} ms |\n"
    f"| Output copy or readback | {metrics['output_copy_or_readback_ms']:.2f} ms |"
))

print(json.dumps(report, indent=2))
_ = Path(f"benchmark-{DEVICE}.json").write_text(json.dumps(report, indent=2) + "\n")


| WEBGPU measurement | Time |
| :--- | ---: |
| Load pretrained model | 0.175 s |
| Eager reference (CPU) | 0.368 s |
| Device setup and initial upload | 6.499 s |
| Compile and complete the first prediction | 12.901 s |
| Warmed prediction, resident input (median of 10) | 136.35 ms |
| Output copy or readback | 1.10 ms |

{
  "passed": true,
  "device": "webgpu",
  "input_size": 256,
  "opencv": "5.0.0",
  "cold_first_call_seconds": 12.901,
  "max_abs_error": 7.149577140808105e-05,
  "model_load_seconds": 0.17549999999999955,
  "eager_cpu_reference_seconds": 0.3675999999999995,
  "device_setup_seconds": 6.499300000000002,
  "compile_and_first_prediction_seconds": 12.901,
  "resident_inference_median_ms": 136.34999999999843,
  "output_copy_or_readback_ms": 1.0999999999974364,
  "repeats": 10
}


## Recorded CPU and WebGPU comparison

Both runs use the same model, sample and FP32 inputs. These measurements are
from this machine; performance on your device may differ.


In [8]:
record = json.loads(Path('benchmark-results.json').read_text())
comparison = record['compiled_inference']
cpu, webgpu = comparison['cpu'], comparison['webgpu']
display(Markdown(
    "| Measurement | CPU | WebGPU |\n| :--- | ---: | ---: |\n"
    f"| Compile and first prediction | {cpu['compile_and_first_prediction_seconds']:.2f} s | "
    f"{webgpu['compile_and_first_prediction_seconds']:.2f} s |\n"
    f"| Warmed prediction, resident input | {cpu['resident_inference_median_ms']:.2f} ms | "
    f"{webgpu['resident_inference_median_ms']:.2f} ms |\n"
    f"| Output copy or readback | {cpu['output_copy_or_readback_ms']:.2f} ms | "
    f"{webgpu['output_copy_or_readback_ms']:.2f} ms |"
))


| Measurement | CPU | WebGPU |
| :--- | ---: | ---: |
| Compile and first prediction | 12.12 s | 12.90 s |
| Warmed prediction, resident input | 335.45 ms | 136.35 ms |
| Output copy or readback | 0.20 ms | 1.10 ms |